# Cycle 1 — Modelling (Chronological Split)

In this notebook - we evaluate only untuned models using a chronological split. This allows for:

1. Measure how well models generalise to future unseen data
2. Compare performance against the optimistic random-split results
3. Identify the true real-world performance of each model

In [3]:
import sys, os              
import pandas as pd         
import numpy as np          
import warnings
warnings.filterwarnings('ignore') 

from sklearn.preprocessing import StandardScaler           # z-score scaler for Logistic Regression
from sklearn.dummy import DummyClassifier                  # always-predict-majority baseline
from sklearn.linear_model import LogisticRegression        # linear classifier
from sklearn.ensemble import RandomForestClassifier        # ensemble of decision trees
from sklearn.metrics import accuracy_score, classification_report  # evaluation metrics
from xgboost import XGBClassifier                          # gradient boosting (XGBoost)
from lightgbm import LGBMClassifier                        # gradient boosting (LightGBM)


_here = os.getcwd()                                     
while not os.path.isdir(os.path.join(_here, 'data')):   
    _p = os.path.dirname(_here)                         
    if _p == _here: raise RuntimeError('project root not found')
    _here = _p
if _here not in sys.path:
    sys.path.insert(0, _here)                             

from config import Paths, ensure_dirs 
ensure_dirs()

print('All libraries imported successfully')


All libraries imported successfully


## Data Prep — Chronological Split

**Hypothesis:** Slightly lower accuracy than random split — the model is now evaluated on genuinely unseen future seasons.

In [5]:
# YOU ARE MISSING THIS LINE:
df1 = pd.read_csv(str(Paths.PL_MATCHES_PROCESSED))  # load preprocessed dataset
# STEP 1: Sort all rows by the 'Season' column from oldest (2000) to newest (2018)
df1 = df1.sort_values('Season').reset_index(drop=True)  
# STEP 2: Find the exact row index that represents the 80% mark of the dataset
test_frac = 0.2
split_idx = int(len(df1) * (1 - test_frac))  
# STEP 3: Slice the dataset down the middle at that 80% mark
train_df1 = df1.iloc[:split_idx]  
test_df1  = df1.iloc[split_idx:]  

X1_train = train_df1.drop(columns=['FTR'])   # features for training (Season kept as a feature here)
y1_train = train_df1['FTR']                  # target for training
X1_test  = test_df1.drop(columns=['FTR'])    # features for test
y1_test  = test_df1['FTR']                   # target for test

scaler1 = StandardScaler()                    # z-score scaler for Logistic Regression
X1_train_s = scaler1.fit_transform(X1_train)  # fit on training only — no leakage
X1_test_s  = scaler1.transform(X1_test)       # apply same transform to test

print(f'Train seasons: {train_df1["Season"].min()} -> {train_df1["Season"].max()}')
print(f'Test  seasons: {test_df1["Season"].min()} -> {test_df1["Season"].max()}')
print(f'Train rows: {len(X1_train)} | Test rows: {len(X1_test)}')
print()
print('Test target distribution:')
print(y1_test.value_counts().sort_index())
print('(0=Away Win, 1=Draw, 2=Home Win)')


Train seasons: 2000 -> 2014
Test  seasons: 2014 -> 2017
Train rows: 5472 | Test rows: 1368

Test target distribution:
FTR
0    406
1    348
2    614
Name: count, dtype: int64
(0=Away Win, 1=Draw, 2=Home Win)


### Model 1: Dummy Classifier

In [6]:
dummy1 = DummyClassifier(strategy='most_frequent', random_state=42)  # always predict Home Win
dummy1.fit(X1_train, y1_train)                                         # just counts class frequency
y_pred_dummy1 = dummy1.predict(X1_test)                                # predicts Home Win for every row

print('DUMMY CLASSIFIER (chronological)')
print(f'Accuracy: {accuracy_score(y1_test, y_pred_dummy1)*100:.2f}%')  # floor accuracy
print(classification_report(y1_test, y_pred_dummy1, target_names=['Away Win','Draw','Home Win']))


DUMMY CLASSIFIER (chronological)
Accuracy: 44.88%
              precision    recall  f1-score   support

    Away Win       0.00      0.00      0.00       406
        Draw       0.00      0.00      0.00       348
    Home Win       0.45      1.00      0.62       614

    accuracy                           0.45      1368
   macro avg       0.15      0.33      0.21      1368
weighted avg       0.20      0.45      0.28      1368



### Model 2: Logistic Regression


In [7]:
lr1 = LogisticRegression(
    max_iter=1000,           # ensure convergence on this feature set
    class_weight='balanced', # up-weight minority classes
    random_state=42
)
lr1.fit(X1_train_s, y1_train)         # train on z-scored features
y_pred_lr1 = lr1.predict(X1_test_s)  # predict on scaled test features

print('LOGISTIC REGRESSION (chronological)')
print(f'Accuracy: {accuracy_score(y1_test, y_pred_lr1)*100:.2f}%')
print(classification_report(y1_test, y_pred_lr1, target_names=['Away Win','Draw','Home Win']))


LOGISTIC REGRESSION (chronological)
Accuracy: 48.76%
              precision    recall  f1-score   support

    Away Win       0.44      0.58      0.50       406
        Draw       0.30      0.20      0.24       348
    Home Win       0.60      0.59      0.59       614

    accuracy                           0.49      1368
   macro avg       0.45      0.46      0.45      1368
weighted avg       0.48      0.49      0.48      1368



### Model 3: Random Forest

In [8]:
rf1 = RandomForestClassifier(
    n_estimators=100,        # 100 trees — stable ensemble
    class_weight='balanced', # compensate for class imbalance
    random_state=42
)
rf1.fit(X1_train, y1_train)          # raw features — trees don't need scaling
y_pred_rf1 = rf1.predict(X1_test)

print('RANDOM FOREST (chronological)')
print(f'Accuracy: {accuracy_score(y1_test, y_pred_rf1)*100:.2f}%')
print(classification_report(y1_test, y_pred_rf1, target_names=['Away Win','Draw','Home Win']))


RANDOM FOREST (chronological)
Accuracy: 51.17%
              precision    recall  f1-score   support

    Away Win       0.51      0.44      0.47       406
        Draw       0.24      0.06      0.10       348
    Home Win       0.54      0.82      0.65       614

    accuracy                           0.51      1368
   macro avg       0.43      0.44      0.41      1368
weighted avg       0.45      0.51      0.46      1368



### Model 4: XGBoost

In [9]:
xgb1 = XGBClassifier(
    n_estimators=100,        # 100 boosting rounds
    random_state=42,
    eval_metric='mlogloss',  # multi-class log loss
    verbosity=0              # suppress output
)
xgb1.fit(X1_train, y1_train)
y_pred_xgb1 = xgb1.predict(X1_test)

print('XGBOOST (chronological)')
print(f'Accuracy: {accuracy_score(y1_test, y_pred_xgb1)*100:.2f}%')
print(classification_report(y1_test, y_pred_xgb1, target_names=['Away Win','Draw','Home Win']))


XGBOOST (chronological)
Accuracy: 48.17%
              precision    recall  f1-score   support

    Away Win       0.43      0.42      0.43       406
        Draw       0.30      0.15      0.20       348
    Home Win       0.55      0.71      0.62       614

    accuracy                           0.48      1368
   macro avg       0.42      0.43      0.41      1368
weighted avg       0.45      0.48      0.45      1368



### Model 5: LightGBM

In [34]:
lgb1 = LGBMClassifier(
    n_estimators=100,  # 100 boosting rounds — same as XGBoost for fair comparison
    random_state=42,
    verbose=-1         # suppress LightGBM training output
)
lgb1.fit(X1_train, y1_train)
y_pred_lgb1 = lgb1.predict(X1_test)

print('LIGHTGBM (chronological)')
print(f'Accuracy: {accuracy_score(y1_test, y_pred_lgb1)*100:.2f}%')
print(classification_report(y1_test, y_pred_lgb1, target_names=['Away Win','Draw','Home Win']))


LIGHTGBM (chronological)
Accuracy: 49.85%
              precision    recall  f1-score   support

    Away Win       0.46      0.48      0.47       406
        Draw       0.28      0.11      0.15       348
    Home Win       0.56      0.74      0.63       614

    accuracy                           0.50      1368
   macro avg       0.43      0.44      0.42      1368
weighted avg       0.46      0.50      0.46      1368



### Results Summary — Chronological vs Random Split

- Compares accuracy of all 5 models under chronological vs random splitting side-by-side.
- Quantifies the effect of data leakage from random splitting. A large positive Delta (random − chrono) signals that the random split was overly optimistic.


In [35]:
results_d1 = pd.DataFrame({                 # build comparison dataframe
    'Model': ['Dummy','Logistic Regression','Random Forest','XGBoost','LightGBM'],
    'Chrono Acc %': [
        accuracy_score(y1_test, y_pred_dummy1)*100,   # dummy (chrono)
        accuracy_score(y1_test, y_pred_lr1)*100,      # logistic regression (chrono)
        accuracy_score(y1_test, y_pred_rf1)*100,      # random forest (chrono)
        accuracy_score(y1_test, y_pred_xgb1)*100,     # xgboost (chrono)
        accuracy_score(y1_test, y_pred_lgb1)*100      # lightgbm (chrono)
    ],
    'Random Acc % (original)': [46.35, 49.85, 51.39, 50.95, 54.02],  # from random-split notebook
})
results_d1['Delta'] = (results_d1['Chrono Acc %'] - results_d1['Random Acc % (original)']).round(2)
# Delta: positive = chrono better than random; negative = random was inflated by future leakage
results_d1['Chrono Acc %'] = results_d1['Chrono Acc %'].round(2)
print('Chronological vs Random split')
print(results_d1.to_string(index=False))


Chronological vs Random split
              Model  Chrono Acc %  Random Acc % (original)  Delta
              Dummy         44.88                    46.35  -1.47
Logistic Regression         48.76                    49.85  -1.09
      Random Forest         51.17                    51.39  -0.22
            XGBoost         48.17                    50.95  -2.78
           LightGBM         49.85                    54.02  -4.17
